In [0]:
# ============================================================
# Project  : ABC E-Commerce
# Layer    : Silver
# Notebook : 07_Silver_Products
# Author   : Suriya
# Purpose  : Clean and validate product data
# ============================================================

In [0]:
from pyspark.sql.functions import *

In [0]:
products_df = spark.table("workspace.default.bronze_products")

In [0]:
products_df.show(5)

products_df.printSchema()

print("Total Records :", products_df.count())

In [0]:
null_df = products_df.filter(
    col("product_id").isNull() |
    col("product_name").isNull() |
    col("price").isNull()
)

print("Null Records :", null_df.count())

null_df.show()

In [0]:
valid_df = products_df.filter(
    col("product_id").isNotNull() &
    col("product_name").isNotNull() &
    col("price").isNotNull()
)

In [0]:
duplicate_df = (
    valid_df
    .groupBy("product_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_df.show()

In [0]:
silver_df = valid_df.dropDuplicates(["product_id"])

In [0]:
invalid_price_df = silver_df.filter(col("price") <= 0)

print("Invalid Price Records :", invalid_price_df.count())

invalid_price_df.show()

In [0]:
silver_df = silver_df.filter(col("price") > 0)

In [0]:
silver_df = (
    silver_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("data_source", lit("products.csv"))
)

In [0]:
silver_df.show(5)

silver_df.printSchema()

print("Silver Records :", silver_df.count())

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.silver_products")

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_products
LIMIT 10
""").show()